# Questão 4 — Análise de Clientes

## Objetivo

Identificar os clientes considerados fiéis pela LH Nautical com base em
ticket médio e diversidade de categorias compradas.

### Critérios

- **Faturamento Total:** soma de `orders.total` por cliente;
- **Frequência:** quantidade de pedidos por cliente;
- **Ticket Médio:** faturamento total dividido pela frequência;
- **Diversidade de Categorias:** quantidade de `category_id` distintos comprados;
- **Clientes Elite:** clientes que compraram em pelo menos 13 categorias;
- **Ranking:** os 10 maiores tickets médios;
- **Desempate:** `customer_id` em ordem crescente.

> O faturamento e a frequência são calculados no nível de `orders` antes do
> relacionamento com `order_items`. Essa separação evita duplicar o valor de
> um pedido quando ele possui múltiplos itens.

In [2]:
from pathlib import Path

import duckdb
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

conn = duckdb.connect()

In [3]:
tables = [
    "orders",
    "order_items",
    "product_variants",
    "products",
    "categories",
]

for table in tables:
    csv_path = RAW_DATA_DIR / f"{table}.csv"

    conn.execute(
        f"""
        CREATE OR REPLACE VIEW {table} AS
        SELECT *
        FROM read_csv_auto('{csv_path.as_posix()}');
        """
    )

In [4]:
conn.execute("SHOW TABLES").df()

,name
0,categories
1,order_items
2,orders
3,product_variants
4,products


In [5]:
##ranking
query_elite_customers = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(total) AS faturamento_total,
        COUNT(id) AS frequencia,
        SUM(total) / COUNT(id) AS ticket_medio
    FROM orders
    GROUP BY customer_id
),

customer_diversity AS (
    SELECT
        o.customer_id,
        COUNT(DISTINCT p.category_id) AS diversidade_categorias
    FROM orders AS o
    INNER JOIN order_items AS oi
        ON oi.order_id = o.id
    INNER JOIN product_variants AS pv
        ON pv.id = oi.product_variant_id
    INNER JOIN products AS p
        ON p.id = pv.product_id
    GROUP BY o.customer_id
)

SELECT
    cs.customer_id,
    ROUND(cs.faturamento_total, 2) AS faturamento_total,
    cs.frequencia,
    ROUND(cs.ticket_medio, 2) AS ticket_medio,
    cd.diversidade_categorias
FROM customer_sales AS cs
INNER JOIN customer_diversity AS cd
    ON cd.customer_id = cs.customer_id
WHERE cd.diversidade_categorias >= 13
ORDER BY
    cs.ticket_medio DESC,
    cs.customer_id ASC
LIMIT 10;
"""

elite_customers = conn.execute(query_elite_customers).df()

elite_customers

,customer_id,faturamento_total,frequencia,ticket_medio,diversidade_categorias
0,22,1087838.44,26,41839.94,14
1,1477,916262.58,22,41648.30,14
2,929,1082775.89,26,41645.23,14
3,1116,655737.20,16,40983.58,14
4,1691,815471.30,20,40773.56,14
5,774,726127.99,18,40340.44,14
6,1470,1040553.09,26,40021.27,14
7,1599,997616.46,25,39904.66,14
8,965,677297.78,17,39841.05,14
9,1722,1146455.22,29,39532.94,14


In [6]:
query_top_category = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(total) AS faturamento_total,
        COUNT(id) AS frequencia,
        SUM(total) / COUNT(id) AS ticket_medio
    FROM orders
    GROUP BY customer_id
),

customer_diversity AS (
    SELECT
        o.customer_id,
        COUNT(DISTINCT p.category_id) AS diversidade_categorias
    FROM orders AS o
    INNER JOIN order_items AS oi
        ON oi.order_id = o.id
    INNER JOIN product_variants AS pv
        ON pv.id = oi.product_variant_id
    INNER JOIN products AS p
        ON p.id = pv.product_id
    GROUP BY o.customer_id
),

elite_customers AS (
    SELECT
        cs.customer_id,
        cs.ticket_medio
    FROM customer_sales AS cs
    INNER JOIN customer_diversity AS cd
        ON cd.customer_id = cs.customer_id
    WHERE cd.diversidade_categorias >= 13
    ORDER BY
        cs.ticket_medio DESC,
        cs.customer_id ASC
    LIMIT 10
)

SELECT
    c.id AS category_id,
    c.name AS categoria,
    SUM(oi.quantity) AS quantidade_total
FROM elite_customers AS ec
INNER JOIN orders AS o
    ON o.customer_id = ec.customer_id
INNER JOIN order_items AS oi
    ON oi.order_id = o.id
INNER JOIN product_variants AS pv
    ON pv.id = oi.product_variant_id
INNER JOIN products AS p
    ON p.id = pv.product_id
INNER JOIN categories AS c
    ON c.id = p.category_id
GROUP BY
    c.id,
    c.name
ORDER BY
    quantidade_total DESC,
    c.id ASC
LIMIT 1;
"""

top_category = conn.execute(query_top_category).df()

top_category

,category_id,categoria,quantidade_total
0,8,Hélices,492.0
